# Notebook 07 — Continual Learning on the CNN Encoder

This notebook implements the class-incremental continual learning experiment from
Milani WIFS 2024 on the whuGAIT CNN encoder (Zou et al. 2020).

**Task structure** — 98 member subjects (IDs 21–118) are split into 5 sequential tasks:

| Task | Subject IDs   | Classes   |
|------|---------------|-----------|
| 0    | 21–40         | 0–19      |
| 1    | 41–60         | 20–39     |
| 2    | 61–80         | 40–59     |
| 3    | 81–100        | 60–79     |
| 4    | 101–118       | 80–97     |

**Methods compared**:
- **Std** — naive fine-tuning (baseline, catastrophic forgetting expected)
- **CDML** — Code Division Modulation Layer (Milani WIFS 2024): each task's
  embedding is modulated by a task-specific ±1 binary code before the FC head.

**Measurements after each task**:
1. Identification accuracy on Task-0 subjects (forgetting curve)
2. MIA AUC on Task-0 subjects vs held-out non-members (privacy curve)

The MIA uses the simple confidence-based attack from the paper: mean softmax
probability on the subject's correct class.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, json, copy, logging, time
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

sys.path.insert(0, '..')
from src.models.gait_cnn   import GaitCNN
from src.models.cl_cnn     import CDMLGaitCNN
from src.training.cl_trainer import CLTrainer
from src.data.dataset      import load_dataset, filter_subjects, normalize

# ── paths ──────────────────────────────────────────────────────────────────
DATA_ROOT    = Path('../data')
ARTIFACT_DIR = Path('../artifacts/whuGAIT')
CKPT_DIR     = Path('../checkpoints/whuGAIT')
RESULT_DIR   = Path('../results/whuGAIT')
LOG_DIR      = Path('../logs/whuGAIT')
for d in [RESULT_DIR, LOG_DIR, CKPT_DIR / 'cl']:
    d.mkdir(parents=True, exist_ok=True)

# ── logging ────────────────────────────────────────────────────────────────
log = logging.getLogger('nb07')
log.setLevel(logging.DEBUG)
log.handlers.clear()
fh = logging.FileHandler(LOG_DIR / '07_cl_cnn.log', mode='w')
fh.setFormatter(logging.Formatter('%(asctime)s  %(message)s', datefmt='%H:%M:%S'))
sh = logging.StreamHandler()
sh.setFormatter(logging.Formatter('%(message)s'))
log.addHandler(fh)
log.addHandler(sh)

# ── config ─────────────────────────────────────────────────────────────────
EPOCHS_PER_TASK = 400   # paper: 400
LR              = 1e-3  # paper: 0.001
LR_GAMMA        = 0.9   # paper: 0.9 exponential decay
BATCH_SIZE      = 512
DEVICE          = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_TOTAL_CLASSES = 98
TASK_SIZE       = 20
SEED_BASE       = 1000  # seed for CDML code generation; seed_k = SEED_BASE + k

log.info(f'Device: {DEVICE}  |  Epochs/task: {EPOCHS_PER_TASK}  |  Tasks: 5')

## 1. Load Data and Define Tasks

In [ ]:
with open(ARTIFACT_DIR / 'subject_split.json') as f:
    split = json.load(f)

member_ids     = sorted(split['train_ids'])       # 98 subjects, IDs 21–118
nonmember_ids  = sorted(split['held_out_ids'])    # 20 subjects, IDs 1–20

# Map original IDs → global class indices 0..97
id_to_cls = {sid: i for i, sid in enumerate(member_ids)}

# 5 tasks of 20 subjects each (last task has 18)
tasks = []
for start in range(0, N_TOTAL_CLASSES, TASK_SIZE):
    end = min(start + TASK_SIZE, N_TOTAL_CLASSES)
    tasks.append({
        'task_id':    len(tasks),
        'subject_ids': member_ids[start:end],
        'class_range': (start, end),     # global class indices
    })

print(f'{len(tasks)} tasks:')
for t in tasks:
    print(f"  Task {t['task_id']}: IDs {t['subject_ids'][0]}–{t['subject_ids'][-1]}"
          f"  classes {t['class_range'][0]}–{t['class_range'][1]-1}  (n={len(t['subject_ids'])})")

In [ ]:
# Load Dataset #1 (all members + non-members)
X_raw_tr, y_raw_tr = load_dataset(str(DATA_ROOT / 'Dataset #1'), 'train')
X_raw_te, y_raw_te = load_dataset(str(DATA_ROOT / 'Dataset #1'), 'test')

# Normalise using member train split stats (no leakage)
from src.data.dataset import filter_subjects
X_mem_tr, y_mem_tr = filter_subjects(X_raw_tr, y_raw_tr, member_ids)
_, norm_stats = normalize(X_mem_tr)

X_all_tr, _ = normalize(X_raw_tr, norm_stats)
X_all_te, _ = normalize(X_raw_te, norm_stats)

def make_loader(X, y_orig, subject_ids, cls_map=None, shuffle=True):
    """Build a DataLoader for a subset of subjects.
    cls_map: if provided, remap original subject IDs to class indices.
    """
    mask = np.isin(y_orig, subject_ids)
    X_s = X[mask]
    y_s = y_orig[mask]
    if cls_map:
        y_s = np.array([cls_map[sid] for sid in y_s], dtype=np.int64)
    else:
        y_s = y_s.astype(np.int64)
    Xt = torch.from_numpy(X_s).float()
    yt = torch.from_numpy(y_s).long()
    return DataLoader(TensorDataset(Xt, yt), batch_size=BATCH_SIZE, shuffle=shuffle)

# Per-task train/eval loaders (global class indices)
task_train_loaders = [
    make_loader(X_all_tr, y_raw_tr, t['subject_ids'], id_to_cls)
    for t in tasks
]
task_eval_loaders = [
    make_loader(X_all_te, y_raw_te, t['subject_ids'], id_to_cls, shuffle=False)
    for t in tasks
]

# Non-member loader for MIA (Dataset #2, held-out subjects)
X_d2, y_d2 = load_dataset(str(DATA_ROOT / 'Dataset #2'), 'train')
X_d2_norm, _ = normalize(X_d2, norm_stats)
# Non-members don't have valid class indices — use placeholder 0
nm_loader = make_loader(X_d2_norm, y_d2, nonmember_ids,
                        cls_map={sid: 0 for sid in nonmember_ids}, shuffle=False)

print(f'Task loaders built. Task-0 train: {len(task_train_loaders[0].dataset)} windows')
print(f'Non-member loader: {len(nm_loader.dataset)} windows from {len(nonmember_ids)} subjects')

## 2. Train — Std (Naive Fine-Tuning)

Baseline: train sequentially on each task with no forgetting protection.
Expected to show catastrophic forgetting on Task-0 accuracy and
high MIA AUC (membership signal persists in weights).

In [ ]:
def run_cl(use_cdml: bool, label: str):
    """Run the full CL sequence. Returns list of TaskResult."""
    model = CDMLGaitCNN(n_classes=N_TOTAL_CLASSES).to(DEVICE) if use_cdml \
            else GaitCNN(n_classes=N_TOTAL_CLASSES).to(DEVICE)

    trainer = CLTrainer(
        model=model, device=DEVICE,
        epochs_per_task=EPOCHS_PER_TASK, lr=LR, lr_gamma=LR_GAMMA,
        use_cdml=use_cdml, seed_base=SEED_BASE,
    )

    results = []
    for t in tasks:
        tid = t['task_id']
        log.info(f'--- [{label}] Training Task {tid} ---')
        print(f'[{label}] Task {tid}...', end=' ', flush=True)

        # eval loaders for all tasks seen so far
        eval_loaders = {i: task_eval_loaders[i] for i in range(tid + 1)}

        r = trainer.train_task(
            task_id        = tid,
            train_loader   = task_train_loaders[tid],
            eval_loaders   = eval_loaders,
            mia_loader_task0    = task_eval_loaders[0],
            mia_nonmember_loader = nm_loader,
            task0_class_range   = tasks[0]['class_range'],
        )
        results.append(r)

        # Save checkpoint
        ckpt_path = CKPT_DIR / 'cl' / f'{label}_task{tid}.pt'
        torch.save(model.state_dict(), ckpt_path)

        acc0 = r.per_task_acc.get(0, float('nan'))
        print(f'acc_task0={acc0*100:.1f}%  mia_auc={r.mia_auc_task0:.3f}  ({r.elapsed_s:.0f}s)')

    return results


results_std = run_cl(use_cdml=False, label='std')

## 3. Train — CDML

In [ ]:
results_cdml = run_cl(use_cdml=True, label='cdml')

## 4. Results — Forgetting and MIA Curves

In [ ]:
n_tasks = len(tasks)
task_nums = list(range(n_tasks))

acc0_std  = [r.per_task_acc.get(0, float('nan')) for r in results_std]
acc0_cdml = [r.per_task_acc.get(0, float('nan')) for r in results_cdml]
mia_std   = [r.mia_auc_task0 for r in results_std]
mia_cdml  = [r.mia_auc_task0 for r in results_cdml]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Forgetting
ax = axes[0]
ax.plot(task_nums, [a*100 for a in acc0_std],  'o-', color='#e74c3c', label='Std (naive)')
ax.plot(task_nums, [a*100 for a in acc0_cdml], 's-', color='#3498db', label='CDML')
ax.axhline(100/N_TOTAL_CLASSES*100, color='grey', linestyle=':', linewidth=1, label='Random baseline')
ax.set_xlabel('CL Task')
ax.set_ylabel('Task-0 Accuracy (%)')
ax.set_title('Catastrophic Forgetting — Task-0 Accuracy')
ax.set_xticks(task_nums)
ax.set_xticklabels([f'After T{i}' for i in task_nums])
ax.legend()
ax.grid(True, alpha=0.3)

# MIA
ax = axes[1]
ax.plot(task_nums, mia_std,  'o-', color='#e74c3c', label='Std (naive)')
ax.plot(task_nums, mia_cdml, 's-', color='#3498db', label='CDML')
ax.axhline(0.5, color='grey', linestyle=':', linewidth=1, label='Random baseline')
ax.set_xlabel('CL Task')
ax.set_ylabel('MIA AUC (Task-0 vs non-members)')
ax.set_title('MIA Privacy — Confidence-Based Attack AUC')
ax.set_xticks(task_nums)
ax.set_xticklabels([f'After T{i}' for i in task_nums])
ax.set_ylim(0.0, 1.05)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(RESULT_DIR / '07_cl_forgetting_mia.png', dpi=150)
plt.show()
log.info('Figure saved: 07_cl_forgetting_mia.png')

## 5. Per-Task Accuracy Matrix

Full forgetting matrix: accuracy on task $i$ after training through task $j \geq i$.

In [ ]:
for label, results in [('Std', results_std), ('CDML', results_cdml)]:
    print(f'\n{label} — accuracy matrix (rows = after task j, cols = evaluated on task i)')
    header = '        ' + '  '.join(f'T{i}' for i in range(n_tasks))
    print(header)
    for r in results:
        row = f'After T{r.task_id}:  '
        for i in range(n_tasks):
            if i in r.per_task_acc:
                row += f'{r.per_task_acc[i]*100:5.1f}  '
            else:
                row += '  --   '
        print(row)

## 6. t-SNE — Embedding Space After Final Task

Reproduces Fig. 3 of Milani WIFS 2024: t-SNE projections of the latent
representation for Std vs CDML after all 5 tasks. CDML is expected to show
more homogeneous distribution across task clusters.

In [ ]:
from sklearn.manifold import TSNE

def extract_embeddings_cl(model, X_np, task_id=None):
    """Extract 2048-dim embeddings (pre-modulation)."""
    model.eval()
    embs = []
    with torch.no_grad():
        for i in range(0, len(X_np), BATCH_SIZE):
            xb = torch.from_numpy(X_np[i:i+BATCH_SIZE]).float().to(DEVICE)
            embs.append(model.encode(xb).cpu().numpy())
    return np.concatenate(embs, axis=0)

# Reload final checkpoints
model_std_final  = GaitCNN(n_classes=N_TOTAL_CLASSES).to(DEVICE)
model_cdml_final = CDMLGaitCNN(n_classes=N_TOTAL_CLASSES).to(DEVICE)
model_std_final.load_state_dict(
    torch.load(CKPT_DIR / 'cl' / f'std_task{n_tasks-1}.pt', map_location=DEVICE))
model_cdml_final.load_state_dict(
    torch.load(CKPT_DIR / 'cl' / f'cdml_task{n_tasks-1}.pt', map_location=DEVICE))

# Sample 2000 windows across all tasks for t-SNE
X_all_te_mem, y_all_te_mem = filter_subjects(X_all_te, y_raw_te, member_ids)
rng = np.random.default_rng(42)
idx = rng.choice(len(X_all_te_mem), size=min(2000, len(X_all_te_mem)), replace=False)
X_samp = X_all_te_mem[idx]
y_samp = np.array([id_to_cls[sid] for sid in y_all_te_mem[idx]], dtype=int)

# Assign task colours (0–19 → T0, 20–39 → T1, …)
task_labels = y_samp // TASK_SIZE
task_labels = np.clip(task_labels, 0, n_tasks - 1)
colours = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6']

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, model, label in [
    (axes[0], model_std_final,  'Std (naive)'),
    (axes[1], model_cdml_final, 'CDML'),
]:
    emb = extract_embeddings_cl(model, X_samp)
    proj = TSNE(n_components=2, perplexity=40, random_state=42).fit_transform(emb)
    for tid in range(n_tasks):
        mask = task_labels == tid
        ax.scatter(proj[mask, 0], proj[mask, 1],
                   c=colours[tid], s=6, alpha=0.6, label=f'Task {tid}')
    ax.set_title(f't-SNE Embeddings — {label}')
    ax.axis('off')
    ax.legend(markerscale=2, fontsize=8)

plt.tight_layout()
plt.savefig(RESULT_DIR / '07_cl_tsne.png', dpi=150)
plt.show()
log.info('Figure saved: 07_cl_tsne.png')

## 7. Summary Table and LaTeX Metrics

In [ ]:
print('=== CL Summary ===')
print(f'{"Method":8s}  {"After Task":10s}  {"T0 Acc":8s}  {"MIA AUC":8s}')
print('-' * 42)
for label, results in [('Std', results_std), ('CDML', results_cdml)]:
    for r in results:
        print(f'{label:8s}  Task {r.task_id:5d}     '
              f'{r.per_task_acc.get(0, float("nan"))*100:6.1f}%  '
              f'{r.mia_auc_task0:.4f}')

# Final-task values for LaTeX
final_std  = results_std[-1]
final_cdml = results_cdml[-1]

from src.utils.latex_writer import write_latex_metrics
TEX_DIR = Path('../latex/generated')
write_latex_metrics('nb07_cl', {
    'clNTasks':            n_tasks,
    'clTaskSize':          TASK_SIZE,
    'clEpochsPerTask':     EPOCHS_PER_TASK,
    # Std final
    'clStdFinalAccTask0':  f'{final_std.per_task_acc.get(0, float("nan"))*100:.1f}\\%',
    'clStdFinalMiaAUC':    f'{final_std.mia_auc_task0:.4f}',
    # CDML final
    'clCdmlFinalAccTask0': f'{final_cdml.per_task_acc.get(0, float("nan"))*100:.1f}\\%',
    'clCdmlFinalMiaAUC':   f'{final_cdml.mia_auc_task0:.4f}',
    # Forgetting = drop in Task-0 acc from after T0 to after T4
    'clStdForgetting':     f'{(acc0_std[0] - acc0_std[-1])*100:.1f}\\%',
    'clCdmlForgetting':    f'{(acc0_cdml[0] - acc0_cdml[-1])*100:.1f}\\%',
}, output_dir=str(TEX_DIR), key_prefix='whuGAIT', log=log)
print(f'LaTeX metrics written to {TEX_DIR}/nb07_cl_metrics.tex')